In [17]:
# packages
import duckdb
import pandas as pd
from topic_model import load_model

In [18]:
# load data
con = duckdb.connect("../guardian_articles.duckdb")
df_topics = con.execute(
    """
    SELECT * from article_topics
    """
).df()

# con.close()
topic_model = load_model()
topic_info = topic_model.get_topic_info().set_index("Topic")

2026-04-12 20:33:15,987 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [19]:
# build table for reporting
total_articles = len(df_topics)

rows = []
for topic_id, row in topic_info.iterrows():
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = ", ".join([w for w, _ in words[:8]])
    else:
        top_words = "—"

    count = int(row["Count"])
    share = count / total_articles * 100

    rows.append(
        {
            "topic_id": topic_id,
            "auto_label": row["Name"],
            "top_words": top_words,
            "count": count,
            "share_pct": round(share, 2),
        }
    )

df_report = (
    pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
)

# Top 20 including -1
df_report = df_report.head(20)

print(f"Total articles: {total_articles:,}\n")
print(df_report.to_string(index=False))

Total articles: 24,660

 topic_id                                                   auto_label                                                                                             top_words  count  share_pct
        0                    0_prime minister_minister_australian_news                         prime minister, minister, australian, news, australia, report, labour, policy  13766      55.82
       -1                      -1_minister_labour_government_political                           minister, labour, government, political, updated gmt, support, news, report   8705      35.30
        1                   1_premier league_liverpool_arsenal_chelsea                       premier league, liverpool, arsenal, chelsea, manchester, united, penalty, match    926       3.76
        2                2_olympic_olympics_gold medal_winter olympics                        olympic, olympics, gold medal, winter olympics, medals, medal, doping, athlete    136       0.55
        3            

In [28]:
# looking further into uncategorized articles
df_outliers = con.execute("""
    SELECT 
        t.id,
        t.topic_id,
        t.topic_prob,
        a.webTitle,
        a.webPublicationDate,
        a.webUrl,
        a.clean_body
    FROM article_topics t
    JOIN cleaned_articles a ON t.id = a.id
    WHERE t.topic_id = -1
    ORDER BY a.webPublicationDate DESC
""").df()


In [29]:
count = df_outliers["webTitle"].str.contains("AI", case=True, na=False).sum()
print(f"Articles with 'AI' in title: {count:,}")

mask = df_outliers["webTitle"].str.contains("AI", case=True, na=False) | df_outliers[
    "clean_body"
].str.contains("AI", case=True, na=False)
print(f"Articles mentioning 'AI' in title or body: {mask.sum():,}")

Articles with 'AI' in title: 1,034
Articles mentioning 'AI' in title or body: 4,419


In [30]:
# label topics
TOPIC_LABEL_OVERRIDES = {
    0: "Government and Politics",
    1: "Football & Soccer",
    2: "Olympics",
    3: "Sleep",
    4: "Trees",
    5: "Classical Music",
    6: "Formula 1",
    7: "Basketball",
    8: "Cosmetic Products",
    9: "Space Exploration",
    10: "Poetry",
    11: "Middle East",
    12: "Catholicism and the Pope",
    13: "Fertility and Reproductive Medicine",
    14: "Nicotine and the Tobacco Industry",
    15: "Crosswords and Puzzles",
    16: "The Beatles",
    17: "Skiing",
    18: "Astronomy",
}


In [31]:
# create labelled table in DuckDB
df_topics["topic_label_clean"] = df_topics["topic_id"].map(
    lambda tid: TOPIC_LABEL_OVERRIDES.get(
        tid, topic_info.loc[tid, "Name"] if tid in topic_info.index else "Uncategorised"
    )
)

# -1 always gets a readable label regardless of overrides
df_topics.loc[df_topics["topic_id"] == -1, "topic_label_clean"] = "Uncategorised"

conn = duckdb.connect("../guardian_articles.duckdb")
conn.execute(
    "CREATE OR REPLACE TABLE article_topics_labelled AS SELECT * FROM df_topics"
)
conn.close()

print(f"Written {len(df_topics):,} rows to article_topics_labelled")
print(f"Overrides applied: {len(TOPIC_LABEL_OVERRIDES)} topics")
print(df_topics["topic_label_clean"].value_counts().head(20).to_string())


Written 24,660 rows to article_topics_labelled
Overrides applied: 19 topics
topic_label_clean
Government and Politics                13766
Uncategorised                           8705
Football & Soccer                        926
Olympics                                 136
Sleep                                    110
Trees                                     80
Classical Music                           47
Formula 1                                 36
Basketball                                34
Cosmetic Products                         34
Space Exploration                         32
Poetry                                    31
Middle East                               30
Catholicism and the Pope                  29
Nicotine and the Tobacco Industry         26
Fertility and Reproductive Medicine       26
Crosswords and Puzzles                    25
The Beatles                               25
Skiing                                    23
Astronomy                                 22
